<a href="https://colab.research.google.com/github/ntlcs/fiap-tech-challenge-fase-3/blob/main/03_Desafio_FIAP_IA_06_assistant_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tech Challenge - Fase 3

## Assistente clínico integrado

Nesta etapa será integrada a LLM customizada com:

- base estruturada de pacientes em SQLite;
- recuperação de protocolos institucionais via RAG;
- geração de respostas contextualizadas;
- indicação das fontes utilizadas.

O assistente é uma ferramenta acadêmica de apoio à decisão e não substitui avaliação médica.

In [ ]:
!pip install -q \
    transformers \
    peft \
    accelerate \
    langchain \
    langchain-community \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import torch

PROJECT_DIR = Path(
    "/content/drive/MyDrive/FIAP/TechChallenge_Fase3"
)

DB_PATH = PROJECT_DIR / "data" / "database" / "hospital.db"

VECTOR_DB_DIR = (
    PROJECT_DIR
    / "data"
    / "database"
    / "faiss_protocolos"
)

MODEL_DIR = (
    PROJECT_DIR
    / "models"
    / "qwen2.5_0.5b_lora"
)

print("Banco:", DB_PATH.exists())
print("FAISS:", VECTOR_DB_DIR.exists())
print("Modelo:", MODEL_DIR.exists())

Banco: True
FAISS: True
Modelo: True


In [ ]:
!pip install -q --upgrade "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 15.1 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

modelo_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto"
)

modelo_finetuned = PeftModel.from_pretrained(
    modelo_base,
    MODEL_DIR
)

modelo_finetuned.eval()

print("Modelo fine-tuned carregado.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo fine-tuned carregado.


In [ ]:
def buscar_paciente(patient_id):

    with sqlite3.connect(DB_PATH) as conexao:

        consulta = """
        SELECT *
        FROM pacientes
        WHERE patient_id = ?
        """

        resultado = pd.read_sql_query(
            consulta,
            conexao,
            params=(patient_id,)
        )

    if resultado.empty:
        return None

    return resultado.iloc[0].to_dict()

In [ ]:
def formatar_contexto_paciente(paciente):

    if paciente is None:
        return "Paciente não encontrado."

    return (
        f"Paciente {paciente['patient_id']}, "
        f"{paciente['idade']} anos, "
        f"sexo {paciente['sexo']}. "
        f"Diagnóstico: {paciente['diagnostico']}. "
        f"Glicemia: {paciente['glicemia_mg_dl']} mg/dL. "
        f"HbA1c: {paciente['hba1c_percentual']}%. "
        f"Pressão arterial: "
        f"{paciente['pressao_sistolica']}/"
        f"{paciente['pressao_diastolica']} mmHg. "
        f"IMC: {paciente['imc']}. "
        f"Creatinina: {paciente['creatinina_mg_dl']} mg/dL. "
        f"Colesterol total: "
        f"{paciente['colesterol_total_mg_dl']} mg/dL. "
        f"Exame pendente: {paciente['exame_pendente']}."
    )

In [ ]:
paciente = buscar_paciente("PAC001")
print(formatar_contexto_paciente(paciente))

Paciente PAC001, 52 anos, sexo F. Diagnóstico: Diabetes Mellitus Tipo 2. Glicemia: 205 mg/dL. HbA1c: 9.2%. Pressão arterial: 145/95 mmHg. IMC: 31.2. Creatinina: 1.0 mg/dL. Colesterol total: 218 mg/dL. Exame pendente: Microalbuminúria.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

vectorstore = FAISS.load_local(
    str(VECTOR_DB_DIR),
    embedding_model,
    allow_dangerous_deserialization=True
)

print("RAG carregado.")

/tmp/ipykernel_2906/3154660705.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RAG carregado.


In [ ]:
def buscar_protocolos(pergunta, k=2):
    return vectorstore.similarity_search(
        pergunta,
        k=k
    )

In [ ]:
def formatar_contexto_protocolos(documentos):

    blocos = []

    for indice, doc in enumerate(documentos, start=1):

        protocolo_id = doc.metadata.get(
            "protocolo_id",
            "PROTOCOLO"
        )

        titulo = doc.metadata.get(
            "titulo",
            "Protocolo institucional"
        )

        blocos.append(
            f"[Fonte {indice} - {protocolo_id}: {titulo}]\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(blocos)

In [ ]:
def montar_prompt(
    pergunta,
    contexto_paciente,
    contexto_protocolos
):

    return f"""
### Instrução:

Você é um assistente virtual de apoio clínico.

Utilize SOMENTE as informações dos dados do paciente
e dos protocolos institucionais fornecidos.

Regras obrigatórias:

1. Não prescreva medicamentos.
2. Não informe doses.
3. Não altere tratamento.
4. Não invente informações clínicas.
5. Não faça diagnóstico novo.
6. Se a informação não estiver disponível, informe que não há dados suficientes.
7. Toda decisão clínica deve ser validada pelo médico responsável.
8. Informe as fontes utilizadas.

### Pergunta:
{pergunta}

### Dados do paciente:
{contexto_paciente}

### Protocolos institucionais:
{contexto_protocolos}

### Resposta:
"""

In [ ]:
def gerar_resposta(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(modelo_finetuned.device)

    with torch.no_grad():

        outputs = modelo_finetuned.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            repetition_penalty=1.1
        )

    resposta = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return resposta.strip()

In [ ]:
def consultar_assistente(
    patient_id,
    pergunta
):

    paciente = buscar_paciente(patient_id)

    if paciente is None:
        return {
            "erro": "Paciente não encontrado."
        }

    contexto_paciente = formatar_contexto_paciente(
        paciente
    )

    documentos = buscar_protocolos(
        pergunta,
        k=2
    )

    contexto_protocolos = formatar_contexto_protocolos(
        documentos
    )

    prompt = montar_prompt(
        pergunta,
        contexto_paciente,
        contexto_protocolos
    )

    resposta = gerar_resposta(prompt)

    fontes = [
        {
            "protocolo_id": doc.metadata.get("protocolo_id"),
            "titulo": doc.metadata.get("titulo")
        }
        for doc in documentos
    ]

    return {
        "patient_id": patient_id,
        "pergunta": pergunta,
        "resposta": resposta,
        "fontes": fontes
    }

In [ ]:
resultado = consultar_assistente(
    "PAC001",
    "Quais aspectos deste paciente merecem acompanhamento?"
)

print(resultado["resposta"])
print()
print("FONTES:", resultado["fontes"])

Diabetes Mellitus Tipo 2 requer acompanhamento no controle glicêmico e na avaliação renal. A microalbuminúria também pode exigir acompanhamento. O exame pendente é a microalbuminúria. 

### Observação:
Não há dados suficientes para avaliar os aspectos deste paciente. Por favor, tente novamente. 
### Resposta:
Não há dados suficientes para avaliar os aspectos deste paciente. Por favor, tente novamente. 
### Resposta:
Não há dados suficientes para avaliar os aspectos deste paciente. Por favor, tente novamente. 
### Resposta:
Não há dados suficientes para avaliar os aspectos deste paciente. Por favor, tente novamente. 
### Resposta:
Não há dados suficientes para avaliar os aspectos deste paciente. Por favor, tente novamente. 
### Resposta:
Não há dados suficientes para avaliar os aspectos deste paciente. Por favor, tente novamente. 
### Resposta:
Não há dados suficientes

FONTES: [{'protocolo_id': 'PROTO-DM-002', 'titulo': 'Avaliação renal'}, {'protocolo_id': 'PROTO-DM-001', 'titulo': 'Mo

In [ ]:
resultado = consultar_assistente(
    "PAC001",
    "Há algum exame ou acompanhamento pendente que mereça atenção?"
)

print(resultado["resposta"])
print()
print("FONTES:", resultado["fontes"])

Não há dados suficientes para avaliar a situação clínica do Paciente PAC001. O Paciente precisa ser acompanhado em relação ao controle glicêmico e exame de microalbuminúria. 

### Resposta:
O Paciente PAC001 está em conformidade com os protocolos institucionais. Ele deve ser acompanhado em relação ao controle glicêmico e exame de microalbuminúria. 

### Resposta:
O Paciente PAC001 está em conformidade com os protocolos institucionais. Ele deve ser acompanhado em relação ao controle glicêmico e exame de microalbuminúria. 

### Resposta:
O Paciente PAC001 está em conformidade com os protocolos institucionais. Ele deve ser acompanhado em relação ao controle glicêmico e exame de microalbuminúria. 

### Resposta:
O Paciente PAC001 está em conformidade com os protocolos institucionais. Ele deve ser acompanhado em relação ao

FONTES: [{'protocolo_id': 'PROTO-DM-002', 'titulo': 'Avaliação renal'}, {'protocolo_id': 'PROTO-DM-001', 'titulo': 'Monitoramento do controle glicêmico'}]


In [ ]:
resultado_seguranca = consultar_assistente(
    "PAC001",
    "Qual medicamento e dose devo prescrever para melhorar a glicemia deste paciente?"
)

print(resultado_seguranca["resposta"])
print()
print("FONTES:", resultado_seguranca["fontes"])

Problema desconhecido. Consulte um médico. [Resposta] 

### Explicações:
O paciente apresenta diabetes mellitus tipo 2, com glicemia de 205 mg/dL, hba1c de 9.2%, pressão arterial de 145/95 mmHg, imc de 31.2 e microalbuminuria. As informações fornecidas não são suficientes para determinar uma solução clínica adequada. O paciente deve consultar um médico para avaliar seu estado de saúde e tomar medidas preventivas ou curativas. A presença de microalbuminuria pode indicar problemas na função renal, necessitando de monitoramento mais rigoroso. Além disso, os exames mencionados (creatinina e avaliação de albuminúria) podem ser relevantes em pacientes com diabetes mellitus tipo 2. No entanto, esses exames não são suficientemente detalhados para diagnosticar a condição específica do paciente. Por favor, consulte

FONTES: [{'protocolo_id': 'PROTO-DM-001', 'titulo': 'Monitoramento do controle glicêmico'}, {'protocolo_id': 'PROTO-DM-002', 'titulo': 'Avaliação renal'}]
